In [21]:
import os
import numpy as np
import nibabel as nib
from nilearn.input_data import NiftiLabelsMasker
from nilearn.image import clean_img
import pandas as pd
from scipy.io import loadmat
import pandas as pd
from nilearn.image import clean_img, index_img
from nilearn.input_data import NiftiLabelsMasker
from scipy import stats
from itertools import combinations
import gc

from scipy.stats import pearsonr
from numpy.random import default_rng
from sklearn.utils import check_random_state 
from numpy.fft import fft, ifft

# Define paths
shen_atlas_path = "../3_atlases/Shen/shen_2mm_268_parcellation.nii.gz"  

# Initialize the masker with the Shen atlas
masker = NiftiLabelsMasker(labels_img=shen_atlas_path)


In [40]:
def cross_modal_prediction(arousal1_array, fc1_array, arousal2_array, fc2_array,
                           predicted_output_path, observed_output_path, features_output_path,
                           alpha_threshold=0.01, use_mean=True, use_binary_weights=True,
                           normalize_by_total=True, use_absolute=False):
    """
    Perform cross-modal prediction of arousal data using functional connectivity features.

    Parameters:
    - arousal1_array: np.array, Target arousal data (n_subj1, n_TRs1)
    - fc1_array: np.array, Target FC data (n_subj1, n_TRs1, n_features)
    - arousal2_array: np.array, Source arousal data (n_subj2, n_TRs2)
    - fc2_array: np.array, Source FC data (n_subj2, n_TRs2, n_features)
    - predicted_output_path: str, File path to save predicted arousal1 array
    - observed_output_path: str, File path to save observed arousal1 array
    - features_output_path: str, File path to save significant features CSV
    - alpha_threshold: float, Significance threshold for feature selection (default=0.01)
    - use_mean: bool, If True, use mean; if False, use median for feature weights.
    - use_binary_weights: bool, If True, use binary weights (1, -1); if False, use actual weights.
    - normalize_by_total: bool, If True, normalize by total significant features; if False, normalize by separate counts.
    - use_absolute: bool, If True, use absolute value sum for positive and negative features.
    """


    print(f"Shape of Arousal 1 data: {arousal1_array.shape}")
    print(f"Shape of Arousal 1 FC data: {fc1_array.shape}")
    print(f"Shape of Arousal 2 data: {arousal2_array.shape}")
    print(f"Shape of Arousal 2 FC data: {fc2_array.shape}")

    # ---- Step 2: Initialize Storage ---- #
    n_subj1, n_TRs1, n_features = fc1_array.shape
    n_subj2, n_TRs2 = arousal2_array.shape

    predicted_arousal1 = np.full((n_subj1, n_TRs1), np.nan)
    observed_arousal1 = arousal1_array

    # ---- Step 3: Feature Extraction from Arousal 2 ---- #
    print("Extracting features from Arousal 2 data...")
    r_matrix = np.zeros((n_subj2, n_features))

    for subj_idx in range(n_subj2):
        fc2 = fc2_array[subj_idx]
        arousal2 = arousal2_array[subj_idx]

        valid_mask = (~np.isnan(arousal2) & ~np.isnan(fc2).any(axis=1))
        fc2_clean = fc2[valid_mask]
        arousal2_clean = arousal2[valid_mask]

        for feature_idx in range(n_features):
            r, _ = stats.pearsonr(fc2_clean[:, feature_idx], arousal2_clean)
            r_matrix[subj_idx, feature_idx] = r

    # ---- Step 4: One-Sample t-Test Across Arousal 2 Subjects ---- #
    print("Running one-sample t-test on Arousal 2 features...")
    t_values, p_values = stats.ttest_1samp(r_matrix, popmean=0, axis=0, nan_policy='omit')

    # Use mean or median as feature weights
    if use_mean:
        feature_weights = np.mean(r_matrix, axis=0)
    else:
        feature_weights = np.median(r_matrix, axis=0)

    significant_features = np.where(p_values < alpha_threshold)[0]

    # Create binary mask or keep actual weights
    if use_binary_weights:
        binary_mask = np.sign(feature_weights[significant_features])
        weight_mask = binary_mask  # Use only -1, 0, 1
        print(weight_mask)
    else:
        weight_mask = feature_weights[significant_features]  # Keep original values
        print(weight_mask)

    num_positive = np.sum(weight_mask > 0)
    num_negative = np.sum(weight_mask < 0)
    total_features = len(significant_features)

    print(f"{len(significant_features)} significant features selected (p < {alpha_threshold})")
    print(f"Number of positive features: {num_positive}")
    print(f"Number of negative features: {num_negative}")
    print(f"Total number of features: {total_features}")

    # ---- Step 5: Map Features to ROI Pairs ---- #
    print("Mapping significant features to ROI pairs...")
    n_ROIs = int((1 + np.sqrt(1 + 8 * n_features)) / 2)
    roi_pairs = list(combinations(range(1, n_ROIs + 1), 2))
    sig_roi_pairs = [roi_pairs[idx] for idx in significant_features]

    significant_df = pd.DataFrame({
        'ROI1': [pair[0] for pair in sig_roi_pairs],
        'ROI2': [pair[1] for pair in sig_roi_pairs],
        'p_value': p_values[significant_features],
        'feature_weight': feature_weights[significant_features]
    })

    # ---- Step 6: Apply Extracted Features to Arousal 1 Data ---- #
    print("Applying Arousal 2-derived features to Arousal 1 data...")

    for subj_idx in range(n_subj1):
        fc1 = fc1_array[subj_idx][:, significant_features]
        masked_fc1 = fc1 * weight_mask  # Apply weights (binary or real)

        arousal_per_TR = np.full(n_TRs1, np.nan)
        for i in range(n_TRs1):
            fc_values = masked_fc1[i, :]

            if use_absolute:
                sum_positive = np.sum(np.abs(fc_values[weight_mask > 0]))
                sum_negative = np.sum(np.abs(fc_values[weight_mask < 0]))
            else:
                sum_positive = np.sum(fc_values[weight_mask > 0])
                sum_negative = np.sum(fc_values[weight_mask < 0])

            if normalize_by_total:
                divisor = total_features if total_features > 0 else 1
            else:
                divisor_positive = num_positive if num_positive > 0 else 1
                divisor_negative = num_negative if num_negative > 0 else 1

            avg_positive = sum_positive / divisor if normalize_by_total else sum_positive / divisor_positive
            avg_negative = sum_negative / divisor if normalize_by_total else sum_negative / divisor_negative

            arousal_per_TR[i] = avg_positive - avg_negative

        predicted_arousal1[subj_idx, :] = arousal_per_TR

    # ---- Step 7: Save Results ---- #
    print("Saving cross-modal prediction results...")

    # Ensure output directories exist
    os.makedirs(os.path.dirname(predicted_output_path), exist_ok=True)
    os.makedirs(os.path.dirname(observed_output_path), exist_ok=True)
    os.makedirs(os.path.dirname(features_output_path), exist_ok=True)

    np.save(predicted_output_path, predicted_arousal1)
    np.save(observed_output_path, observed_arousal1)
    significant_df.to_csv(features_output_path, index=False)

    print("✅ Cross-modal prediction completed! 🚀")

def predict_sleepwake(arousal2_array, fc2_array,
                           alpha_threshold=0.01, use_mean=True, use_binary_weights=True,
                           normalize_by_total=True, use_absolute=False):
    """
    Perform cross-modal prediction of arousal data using functional connectivity features.

    Parameters:
    - arousal1_array: np.array, Target arousal data (n_subj1, n_TRs1)
    - fc1_array: np.array, Target FC data (n_subj1, n_TRs1, n_features)
    - arousal2_array: np.array, Source arousal data (n_subj2, n_TRs2)
    - fc2_array: np.array, Source FC data (n_subj2, n_TRs2, n_features)
    - predicted_output_path: str, File path to save predicted arousal1 array
    - observed_output_path: str, File path to save observed arousal1 array
    - features_output_path: str, File path to save significant features CSV
    - alpha_threshold: float, Significance threshold for feature selection (default=0.01)
    - use_mean: bool, If True, use mean; if False, use median for feature weights.
    - use_binary_weights: bool, If True, use binary weights (1, -1); if False, use actual weights.
    - normalize_by_total: bool, If True, normalize by total significant features; if False, normalize by separate counts.
    - use_absolute: bool, If True, use absolute value sum for positive and negative features.
    """

    print(f"Shape of Arousal 2 data: {arousal2_array.shape}")
    print(f"Shape of Arousal 2 FC data: {fc2_array.shape}")

    # ---- Step 2: Initialize Storage ---- #
    n_subj2, n_TRs2 = arousal2_array.shape

    # ---- Step 3: Feature Extraction from Arousal 2 ---- #
    print("Extracting features from Arousal 2 data...")
    n_features = fc2_array.shape[2]
    r_matrix = np.zeros((n_subj2, n_features))

    for subj_idx in range(n_subj2):
        fc2 = fc2_array[subj_idx]
        arousal2 = arousal2_array[subj_idx]

        valid_mask = (~np.isnan(arousal2) & ~np.isnan(fc2).any(axis=1))
        fc2_clean = fc2[valid_mask]
        arousal2_clean = arousal2[valid_mask]

        for feature_idx in range(n_features):
            r, _ = stats.pearsonr(fc2_clean[:, feature_idx], arousal2_clean)
            r_matrix[subj_idx, feature_idx] = r

    # ---- Step 4: One-Sample t-Test Across Arousal 2 Subjects ---- #
    print("Running one-sample t-test on Arousal 2 features...")
    t_values, p_values = stats.ttest_1samp(r_matrix, popmean=0, axis=0, nan_policy='omit')

    # Use mean or median as feature weights
    if use_mean:
        feature_weights = np.mean(r_matrix, axis=0)
    else:
        feature_weights = np.median(r_matrix, axis=0)

    significant_features = np.where(p_values < alpha_threshold)[0]

    # Create binary mask or keep actual weights
    if use_binary_weights:
        binary_mask = np.sign(feature_weights[significant_features])
        weight_mask = binary_mask  # Use only -1, 0, 1
        print(weight_mask)
    else:
        weight_mask = feature_weights[significant_features]  # Keep original values
        print(weight_mask)

    total_features = len(significant_features)
    num_positive = np.sum(weight_mask > 0)
    num_negative = np.sum(weight_mask < 0)

    sleepwake_dir = "../4_results/1_arousal/1_EEG_Sleep/combined_sleepwake"
    sleepwake_fc_files = sorted([f for f in os.listdir(sleepwake_dir) if f.endswith("fc.npy")])
    arousal_files = sorted([f for f in os.listdir(sleepwake_dir) if f.endswith("wake.npy")])

    fc_subjects = {
        f.replace('_combined_fc.npy', '')
        for f in sleepwake_fc_files
    }

    sleepwake_arousal_files = [
        f for f in arousal_files
        if f.replace('_wake.npy', '') in fc_subjects
    ]

    all_corrs = []
    predicted_arousal = {}
    observed_arousal = {}

    for i, file in enumerate(sleepwake_fc_files):
        fc1_array = np.load(os.path.join(sleepwake_dir, file))
        sleepwake_arousal_array = np.load(os.path.join(sleepwake_dir, sleepwake_arousal_files[i]))[40:]
        n_TRs1 = sleepwake_arousal_array.shape[0]

        fc1 = fc1_array[40:, significant_features]
        masked_fc1 = fc1 * weight_mask  # Apply weights (binary or real)

        
        arousal_per_TR = np.full(n_TRs1, np.nan)
        for i in range(n_TRs1):
            fc_values = masked_fc1[i, :]

            if use_absolute:
                sum_positive = np.sum(np.abs(fc_values[weight_mask > 0]))
                sum_negative = np.sum(np.abs(fc_values[weight_mask < 0]))
            else:
                sum_positive = np.sum(fc_values[weight_mask > 0])
                sum_negative = np.sum(fc_values[weight_mask < 0])

            if normalize_by_total:
                divisor = total_features if total_features > 0 else 1
            else:
                divisor_positive = num_positive if num_positive > 0 else 1
                divisor_negative = num_negative if num_negative > 0 else 1

            avg_positive = sum_positive / divisor if normalize_by_total else sum_positive / divisor_positive
            avg_negative = sum_negative / divisor if normalize_by_total else sum_negative / divisor_negative

            arousal_per_TR[i] = avg_positive - avg_negative

        if n_TRs1 > 1:

            correlation = pearsonr(arousal_per_TR, sleepwake_arousal_array)[0]
            predicted_arousal[file] = arousal_per_TR
            observed_arousal[file] = sleepwake_arousal_array
            #print(correlation)
            all_corrs.append(correlation)

    z_vals = fisher_z(np.array(all_corrs))
    z_stat = np.nanmean(z_vals)
    mean_corr = fisher_z_inv(z_stat)
    print(f"Mean correlation across all subjects: {mean_corr}")
    p_value = 0

    all_random = []

    for iteration in range(10000):
        randomized_corrs = []
        for i, file in enumerate(sleepwake_fc_files):
            if file in predicted_arousal:
                sleepwake_arousal_array = observed_arousal[file]
                predicte_arousal_array = predicted_arousal[file]
                randomized_arousal = phase_randomize(sleepwake_arousal_array)
                randomized_correlation = pearsonr(predicte_arousal_array, randomized_arousal)[0]
                randomized_corrs.append(randomized_correlation)
        randomized_z_vals = fisher_z(np.array(randomized_corrs))
        randomized_z_stat = np.nanmean(randomized_z_vals)
        randomized_total_corr = fisher_z_inv(randomized_z_stat)
        all_random.append(randomized_total_corr)
        #print(randomized_total_corr)
        if randomized_total_corr >= mean_corr:
            p_value += 1
    p_value /= 10001
    all_random = np.array(all_random)
    print(f"P-value from permutation test: {p_value}")

    return mean_corr, p_value, predicted_arousal, observed_arousal, all_random, all_corrs
            
def compute_subject_feature_correlations(arousal_array, fc_array):
    """
    Compute per-subject correlations between FC features and arousal.

    Parameters:
    - arousal_array: shape (n_subj, n_TRs)
    - fc_array: shape (n_subj, n_TRs, n_features)

    Returns:
    - r_matrix: shape (n_subj, n_features)
    """
    n_subj, _, n_features = fc_array.shape
    r_matrix = np.full((n_subj, n_features), np.nan)

    for subj_idx in range(n_subj):
        arousal = arousal_array[subj_idx]
        fc = fc_array[subj_idx]

        valid_mask = (~np.isnan(arousal) & ~np.isnan(fc).any(axis=1))
        arousal_clean = arousal[valid_mask]
        fc_clean = fc[valid_mask]

        for feat_idx in range(n_features):
            if len(arousal_clean) > 1:
                r, _ = stats.pearsonr(fc_clean[:, feat_idx], arousal_clean)
                r_matrix[subj_idx, feat_idx] = r

    return r_matrix

def fisher_z(r):
    r = np.clip(r, -0.999999, 0.999999)
    return 0.5 * (np.log1p(r) - np.log1p(-r))

def fisher_z_inv(z):
    return (np.exp(2 * z) - 1) / (np.exp(2 * z) + 1)

def phase_randomize(data, random_state=None):
    random_state = check_random_state(random_state)
    data = np.array(data)
    fft_data = fft(data, axis=0)
    if data.shape[0] % 2 == 0:
        pos_freq = np.arange(1, data.shape[0] // 2)
        neg_freq = np.arange(data.shape[0] - 1, data.shape[0] // 2, -1)
    else:
        pos_freq = np.arange(1, (data.shape[0] - 1) // 2 + 1)
        neg_freq = np.arange(data.shape[0] - 1, (data.shape[0] - 1) // 2, -1)
    if len(data.shape) == 1:
        phase_shifts = random_state.uniform(0, 2 * np.pi, size=(len(pos_freq)))
        fft_data[pos_freq] *= np.exp(1j * phase_shifts)
        fft_data[neg_freq] *= np.exp(-1j * phase_shifts)
    else:
        phase_shifts = random_state.uniform(0, 2 * np.pi, size=(len(pos_freq), data.shape[1]))
        fft_data[pos_freq, :] *= np.exp(1j * phase_shifts)
        fft_data[neg_freq, :] *= np.exp(-1j * phase_shifts)
    return np.real(ifft(fft_data, axis=0))


def SR_to_sleepwake(significant_features, weight_mask, use_absolute=False, normalize_by_total=True):

    
    sleepwake_dir = "../4_results/1_arousal/1_EEG_Sleep/combined_sleepwake"
    sleepwake_fc_files = sorted([f for f in os.listdir(sleepwake_dir) if f.endswith("fc.npy")])
    arousal_files = sorted([f for f in os.listdir(sleepwake_dir) if f.endswith("wake.npy")])

    fc_subjects = {
        f.replace('_combined_fc.npy', '')
        for f in sleepwake_fc_files
    }

    sleepwake_arousal_files = [
        f for f in arousal_files
        if f.replace('_wake.npy', '') in fc_subjects
    ]

    total_features = len(significant_features)
    num_positive = np.sum(weight_mask > 0)
    num_negative = np.sum(weight_mask < 0)

    all_corrs = []
    predicted_arousal = {}
    observed_arousal = {}

    for i, file in enumerate(sleepwake_fc_files):
        fc1_array = np.load(os.path.join(sleepwake_dir, file))
        sleepwake_arousal_array = np.load(os.path.join(sleepwake_dir, sleepwake_arousal_files[i]))[40:]
        n_TRs1 = sleepwake_arousal_array.shape[0]

        fc1 = fc1_array[40:, significant_features]
        masked_fc1 = fc1 * weight_mask  # Apply weights (binary or real)

        
        arousal_per_TR = np.full(n_TRs1, np.nan)
        for i in range(n_TRs1):
            fc_values = masked_fc1[i, :]

            if use_absolute:
                sum_positive = np.sum(np.abs(fc_values[weight_mask > 0]))
                sum_negative = np.sum(np.abs(fc_values[weight_mask < 0]))
            else:
                sum_positive = np.sum(fc_values[weight_mask > 0])
                sum_negative = np.sum(fc_values[weight_mask < 0])

            if normalize_by_total:
                divisor = total_features if total_features > 0 else 1
            else:
                divisor_positive = num_positive if num_positive > 0 else 1
                divisor_negative = num_negative if num_negative > 0 else 1

            avg_positive = sum_positive / divisor if normalize_by_total else sum_positive / divisor_positive
            avg_negative = sum_negative / divisor if normalize_by_total else sum_negative / divisor_negative

            arousal_per_TR[i] = avg_positive - avg_negative

        if n_TRs1 > 1:

            correlation = pearsonr(arousal_per_TR, sleepwake_arousal_array)[0]
            predicted_arousal[file] = arousal_per_TR
            observed_arousal[file] = sleepwake_arousal_array
            #print(correlation)
            all_corrs.append(correlation)

    z_vals = fisher_z(np.array(all_corrs))
    z_stat = np.nanmean(z_vals)
    mean_corr = fisher_z_inv(z_stat)
    print(f"Mean correlation across all subjects: {mean_corr}")
    p_value = 0

    all_random = []

    for iteration in range(10000):
        randomized_corrs = []
        for i, file in enumerate(sleepwake_fc_files):
            if file in predicted_arousal:
                sleepwake_arousal_array = observed_arousal[file]
                predicte_arousal_array = predicted_arousal[file]
                randomized_arousal = phase_randomize(sleepwake_arousal_array)
                randomized_correlation = pearsonr(predicte_arousal_array, randomized_arousal)[0]
                randomized_corrs.append(randomized_correlation)
        randomized_z_vals = fisher_z(np.array(randomized_corrs))
        randomized_z_stat = np.nanmean(randomized_z_vals)
        randomized_total_corr = fisher_z_inv(randomized_z_stat)
        all_random.append(randomized_total_corr)
        #print(randomized_total_corr)
        if randomized_total_corr >= mean_corr:
            p_value += 1
    p_value /= 10001
    print(f"P-value from permutation test: {p_value}")
    all_random = np.array(all_random)

    return mean_corr, p_value, predicted_arousal, observed_arousal, all_random, all_corrs

def select_and_save_significant_features_v2(
    arousal_sherlock, fc_sherlock,
    arousal_filmfest, fc_filmfest,
    alpha_threshold=0.01,
    use_mean=True
):
    """
    Computes subject-wise correlations within each dataset, combines them,
    and saves significant features (based on t-test) to CSV.

    Parameters:
    - arousal_sherlock: np.array (n_subj_sher, n_TR_sher)
    - fc_sherlock: np.array (n_subj_sher, n_TR_sher, n_features)
    - arousal_filmfest: np.array (n_subj_film, n_TR_film)
    - fc_filmfest: np.array (n_subj_film, n_TR_film, n_features)
    - output_csv_path: str, path to save CSV
    - alpha_threshold: float, p-value cutoff for significance
    - use_mean: bool, use mean (True) or median (False) for feature weights
    """

    print("Computing per-subject correlations...")
    r_sher = compute_subject_feature_correlations(arousal_sherlock, fc_sherlock)
    r_film = compute_subject_feature_correlations(arousal_filmfest, fc_filmfest)

    print("Combining datasets...")
    r_matrix = np.vstack([r_sher, r_film])  # shape (n_total_subj, n_features)
    n_features = r_matrix.shape[1]

    print("Running one-sample t-test across all subjects...")
    t_values, p_values = stats.ttest_1samp(r_matrix, popmean=0, axis=0, nan_policy='omit')
    feature_weights = np.mean(r_matrix, axis=0) if use_mean else np.median(r_matrix, axis=0)

    significant_features = np.where(p_values < alpha_threshold)[0]
    print(f"{len(significant_features)} features passed p < {alpha_threshold}")

    print("Mapping significant features to ROI pairs...")
    n_ROIs = int((1 + np.sqrt(1 + 8 * n_features)) / 2)
    roi_pairs = list(combinations(range(1, n_ROIs + 1), 2))
    selected_pairs = [roi_pairs[i] for i in significant_features]

    df = pd.DataFrame({
        'ROI1': [pair[0] for pair in selected_pairs],
        'ROI2': [pair[1] for pair in selected_pairs],
        'p_value': p_values[significant_features],
        'feature_weight': feature_weights[significant_features]
    })

    #df.to_csv(output_csv_path, index=False)
    #print(f"✅ Saved {len(df)} significant features to {output_csv_path}")

    return df, significant_features, feature_weights

In [5]:
fmri_dir = "../1_data/1_EEG_Rest_fMRI/"
start_timestamps = list(range(0,571,30))
end_timestamps = list(range(30,601,30))

TR = 2.1  
target_window_seconds = 30  

tr_start_timestamps = np.round(np.array(start_timestamps) / TR)
tr_end_timestamps = np.round(np.array(end_timestamps) / TR)

window_size = max(1, int(target_window_seconds / TR)) 
print(f"Window size: {window_size}")

# Define Gaussian window parameters
sigma = window_size / 4  # Controls decay of Gaussian weights
center = window_size // 2  # Midpoint of the window

# Generate Gaussian weights
indices = np.arange(window_size)
weights = np.exp(-((indices - center) ** 2) / (2 * sigma ** 2))
weights /= np.sum(weights)  # Normalize weights

# Get a list of unique subjects from filenames
fmri_files = sorted([f for f in os.listdir(fmri_dir) if f.endswith(".nii.gz")])

fc_list = []

for file in fmri_files:
    subj = file.split("_")[0]
    if subj != "sub-03" and subj != "sub-09":

        run = ''.join([char for char in file.split("_")[2] if char.isdigit()])
        fmri_file = file
        confound_file = f"{subj}_task-rest_run-{run}_desc-confounds_timeseries.tsv"

        fmri_path = os.path.join(fmri_dir, fmri_file)
        confound_path = os.path.join(fmri_dir, confound_file)

        # Skip if either file is missing
        if not os.path.exists(fmri_path) or not os.path.exists(confound_path):
            print(f"Missing file for {subj} run-{run}, skipping.")
            continue

        print(f"\nProcessing: {fmri_file}")

        # 📌 **Step 1: Load & Prepare Confounds**
        confounds = pd.read_csv(confound_path, sep="\t")

        # Select confounds to regress out
        confound_columns = ["global_signal", "white_matter", "csf",
                            "trans_x", "trans_y", "trans_z", "rot_x", "rot_y", "rot_z"]


        # 📌 **Check for Excessive Motion**
        motion_threshold_mm = 3.0  # Max allowed translation in mm
        rotation_threshold_rad = 3 * (np.pi / 180)  # Convert 3 degrees to radians

        # Extract motion parameters
        motion_params = confounds[["trans_x", "trans_y", "trans_z", "rot_x", "rot_y", "rot_z"]].copy()

        # Compute max absolute motion per run
        max_translation = motion_params[["trans_x", "trans_y", "trans_z"]].abs().max().max()
        max_rotation = motion_params[["rot_x", "rot_y", "rot_z"]].abs().max().max()

        # Flag runs with excessive motion
        if max_translation > motion_threshold_mm or max_rotation > rotation_threshold_rad:
            print(f"⚠️ WARNING: {subj} run-{run} exceeded motion threshold!")
            print(f"   Max Translation: {max_translation:.2f} mm")
            print(f"   Max Rotation: {np.degrees(max_rotation):.2f}°")

        # Ensure selected confounds exist in the file
        confound_matrix = confounds[confound_columns].copy()

        cleaned_img = clean_img(fmri_path, detrend = False, standardize = True, confounds = confound_matrix)
        time_series = masker.fit_transform(X=cleaned_img)
        n_TRs, n_regions = time_series.shape
        n_features = (n_regions * (n_regions - 1)) // 2

        event_fc_vectors = []

        for i in range(len(tr_start_timestamps)):
            start = int(tr_start_timestamps[i])
            end = int(tr_end_timestamps[i])

            if end > n_TRs:
                print(f"⚠️ Event ends beyond time series: {start}-{end} → truncating")
                end = n_TRs

            ts = time_series[start:end]
            if ts.shape[0] < 2:
                event_fc_vectors.append(np.full(n_features, np.nan))
                continue

            cov = np.cov(ts.T)
            std = np.sqrt(np.diag(cov))
            std[std == 0] = 1e-6
            corr = cov / np.outer(std, std)
            fc_vec = corr[np.triu_indices_from(corr, k=1)]
            event_fc_vectors.append(fc_vec)

    fc_list.append(np.stack(event_fc_vectors))  


if len(fc_list) % 2 != 0:
    raise ValueError("⚠️ Each subject must have exactly 2 runs.")

n_subjects = 24
fc_list_2 = np.squeeze(fc_list)
final_fc = np.empty((n_subjects, 40, n_features))
for j in range(n_subjects):
    run1 = fc_list_2[2*j,:,:]
    run2 = fc_list_2[2*j + 1,:,:]
    both_runs = np.concatenate((run1, run2), axis=0)
    final_fc[j,:,:] = both_runs

np.save("../4_results/1_arousal/1_EEG_Sleep/EEG_rest_events.npy", final_fc)

Window size: 14

Processing: sub-02_task-rest_run1_space-MNI152NLin6Asym_desc-filtered_bold.nii.gz


C:\Users\kanno\AppData\Local\Temp\ipykernel_5092\2269608741.py:74: FutureWarning: boolean values for 'standardize' will be deprecated in nilearn 0.15.0.
Use 'zscore_sample' instead of 'True' or use 'None' instead of 'False'.
  cleaned_img = clean_img(fmri_path, detrend = False, standardize = True, confounds = confound_matrix)
C:\Users\kanno\AppData\Local\Temp\ipykernel_5092\2269608741.py:74: DeprecationWarning: From release 0.14.0, confounds will be standardized using the sample std instead of the population std.
  cleaned_img = clean_img(fmri_path, detrend = False, standardize = True, confounds = confound_matrix)
C:\Users\kanno\AppData\Local\Temp\ipykernel_5092\2269608741.py:74: FutureWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the default strategy will be replaced by the new strategy, the 'zscore


Processing: sub-02_task-rest_run2_space-MNI152NLin6Asym_desc-filtered_bold.nii.gz


C:\Users\kanno\AppData\Local\Temp\ipykernel_5092\2269608741.py:74: FutureWarning: boolean values for 'standardize' will be deprecated in nilearn 0.15.0.
Use 'zscore_sample' instead of 'True' or use 'None' instead of 'False'.
  cleaned_img = clean_img(fmri_path, detrend = False, standardize = True, confounds = confound_matrix)
C:\Users\kanno\AppData\Local\Temp\ipykernel_5092\2269608741.py:74: DeprecationWarning: From release 0.14.0, confounds will be standardized using the sample std instead of the population std.
  cleaned_img = clean_img(fmri_path, detrend = False, standardize = True, confounds = confound_matrix)
C:\Users\kanno\AppData\Local\Temp\ipykernel_5092\2269608741.py:74: FutureWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the default strategy will be replaced by the new strategy, the 'zscore


Processing: sub-04_task-rest_run1_space-MNI152NLin6Asym_desc-filtered_bold.nii.gz


C:\Users\kanno\AppData\Local\Temp\ipykernel_5092\2269608741.py:74: FutureWarning: boolean values for 'standardize' will be deprecated in nilearn 0.15.0.
Use 'zscore_sample' instead of 'True' or use 'None' instead of 'False'.
  cleaned_img = clean_img(fmri_path, detrend = False, standardize = True, confounds = confound_matrix)
C:\Users\kanno\AppData\Local\Temp\ipykernel_5092\2269608741.py:74: DeprecationWarning: From release 0.14.0, confounds will be standardized using the sample std instead of the population std.
  cleaned_img = clean_img(fmri_path, detrend = False, standardize = True, confounds = confound_matrix)
C:\Users\kanno\AppData\Local\Temp\ipykernel_5092\2269608741.py:74: FutureWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the default strategy will be replaced by the new strategy, the 'zscore

... (86 outputs left out)


Processing: sub-33_task-rest_run1_space-MNI152NLin6Asym_desc-filtered_bold.nii.gz


C:\Users\kanno\AppData\Local\Temp\ipykernel_5092\2269608741.py:74: FutureWarning: boolean values for 'standardize' will be deprecated in nilearn 0.15.0.
Use 'zscore_sample' instead of 'True' or use 'None' instead of 'False'.
  cleaned_img = clean_img(fmri_path, detrend = False, standardize = True, confounds = confound_matrix)
C:\Users\kanno\AppData\Local\Temp\ipykernel_5092\2269608741.py:74: DeprecationWarning: From release 0.14.0, confounds will be standardized using the sample std instead of the population std.
  cleaned_img = clean_img(fmri_path, detrend = False, standardize = True, confounds = confound_matrix)
C:\Users\kanno\AppData\Local\Temp\ipykernel_5092\2269608741.py:74: FutureWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the default strategy will be replaced by the new strategy, the 'zscore


Processing: sub-33_task-rest_run2_space-MNI152NLin6Asym_desc-filtered_bold.nii.gz


C:\Users\kanno\AppData\Local\Temp\ipykernel_5092\2269608741.py:74: FutureWarning: boolean values for 'standardize' will be deprecated in nilearn 0.15.0.
Use 'zscore_sample' instead of 'True' or use 'None' instead of 'False'.
  cleaned_img = clean_img(fmri_path, detrend = False, standardize = True, confounds = confound_matrix)
C:\Users\kanno\AppData\Local\Temp\ipykernel_5092\2269608741.py:74: DeprecationWarning: From release 0.14.0, confounds will be standardized using the sample std instead of the population std.
  cleaned_img = clean_img(fmri_path, detrend = False, standardize = True, confounds = confound_matrix)
C:\Users\kanno\AppData\Local\Temp\ipykernel_5092\2269608741.py:74: FutureWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the default strategy will be replaced by the new strategy, the 'zscore

In [6]:
fmri_dir = "../1_data/1_EEG_Sleep_fMRI/"

start_timestamps = list(range(0,871,30))
end_timestamps = list(range(30,901,30))

TR = 2.1  
target_window_seconds = 30  

tr_start_timestamps = np.round(np.array(start_timestamps) / TR)
tr_end_timestamps = np.round(np.array(end_timestamps) / TR)

window_size = max(1, int(target_window_seconds / TR)) 
print(f"Window size: {window_size}")

# Define Gaussian window parameters
sigma = window_size / 4  # Controls decay of Gaussian weights
center = window_size // 2  # Midpoint of the window

# Generate Gaussian weights
indices = np.arange(window_size)
weights = np.exp(-((indices - center) ** 2) / (2 * sigma ** 2))
weights /= np.sum(weights)  # Normalize weights

# Get a list of unique subjects from filenames
fmri_files = sorted([f for f in os.listdir(fmri_dir) if f.endswith(".nii.gz")])

fc_list = []

for file in fmri_files:
    subj = file.split("_")[0]

    run = ''.join([char for char in file.split("_")[2] if char.isdigit()])
    fmri_file = file
    confound_file = f"{subj}_task-sleep_run-{run}_desc-confounds_timeseries.tsv"

    fmri_path = os.path.join(fmri_dir, fmri_file)
    confound_path = os.path.join(fmri_dir, confound_file)

    # Skip if either file is missing
    if not os.path.exists(fmri_path) or not os.path.exists(confound_path):
        print(f"Missing file for {subj} run-{run}, skipping.")
        continue

    print(f"\nProcessing: {fmri_file}")

    # 📌 **Step 1: Load & Prepare Confounds**
    confounds = pd.read_csv(confound_path, sep="\t")

    # Select confounds to regress out
    confound_columns = ["global_signal", "white_matter", "csf",
                        "trans_x", "trans_y", "trans_z", "rot_x", "rot_y", "rot_z"]


    # 📌 **Check for Excessive Motion**
    motion_threshold_mm = 3.0  # Max allowed translation in mm
    rotation_threshold_rad = 3 * (np.pi / 180)  # Convert 3 degrees to radians

    # Extract motion parameters
    motion_params = confounds[["trans_x", "trans_y", "trans_z", "rot_x", "rot_y", "rot_z"]].copy()

    # Compute max absolute motion per run
    max_translation = motion_params[["trans_x", "trans_y", "trans_z"]].abs().max().max()
    max_rotation = motion_params[["rot_x", "rot_y", "rot_z"]].abs().max().max()

    # Flag runs with excessive motion
    if max_translation > motion_threshold_mm or max_rotation > rotation_threshold_rad:
        print(f"⚠️ WARNING: {subj} run-{run} exceeded motion threshold!")
        print(f"   Max Translation: {max_translation:.2f} mm")
        print(f"   Max Rotation: {np.degrees(max_rotation):.2f}°")

    # Ensure selected confounds exist in the file
    confound_matrix = confounds[confound_columns].copy()

    cleaned_img = clean_img(fmri_path, detrend = False, standardize = True, confounds = confound_matrix)
    time_series = masker.fit_transform(X=cleaned_img)
    n_TRs, n_regions = time_series.shape
    n_features = (n_regions * (n_regions - 1)) // 2

    event_fc_vectors = []

    for i in range(len(tr_start_timestamps)):
        start = int(tr_start_timestamps[i])
        end = int(tr_end_timestamps[i])

        if end > n_TRs:
            print(f"⚠️ Event ends beyond time series: {start}-{end} → truncating")
            end = n_TRs

        ts = time_series[start:end]
        if ts.shape[0] < 2:
            event_fc_vectors.append(np.full(n_features, np.nan))
            continue

        cov = np.cov(ts.T)
        std = np.sqrt(np.diag(cov))
        std[std == 0] = 1e-6
        corr = cov / np.outer(std, std)
        fc_vec = corr[np.triu_indices_from(corr, k=1)]
        event_fc_vectors.append(fc_vec)

    fc_list.append(np.stack(event_fc_vectors))  

fc_list_2 = np.squeeze(fc_list)
np.save("../4_results/1_arousal/1_EEG_Sleep/EEG_sleep_events.npy", fc_list_2)

Window size: 14

Processing: sub-04_task-sleep_run1_space-MNI152NLin6Asym_desc-filtered_bold.nii.gz


C:\Users\kanno\AppData\Local\Temp\ipykernel_5092\1823858860.py:74: FutureWarning: boolean values for 'standardize' will be deprecated in nilearn 0.15.0.
Use 'zscore_sample' instead of 'True' or use 'None' instead of 'False'.
  cleaned_img = clean_img(fmri_path, detrend = False, standardize = True, confounds = confound_matrix)
C:\Users\kanno\AppData\Local\Temp\ipykernel_5092\1823858860.py:74: DeprecationWarning: From release 0.14.0, confounds will be standardized using the sample std instead of the population std.
  cleaned_img = clean_img(fmri_path, detrend = False, standardize = True, confounds = confound_matrix)
C:\Users\kanno\AppData\Local\Temp\ipykernel_5092\1823858860.py:74: FutureWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the default strategy will be replaced by the new strategy, the 'zscore


Processing: sub-04_task-sleep_run2_space-MNI152NLin6Asym_desc-filtered_bold.nii.gz


C:\Users\kanno\AppData\Local\Temp\ipykernel_5092\1823858860.py:74: FutureWarning: boolean values for 'standardize' will be deprecated in nilearn 0.15.0.
Use 'zscore_sample' instead of 'True' or use 'None' instead of 'False'.
  cleaned_img = clean_img(fmri_path, detrend = False, standardize = True, confounds = confound_matrix)
C:\Users\kanno\AppData\Local\Temp\ipykernel_5092\1823858860.py:74: DeprecationWarning: From release 0.14.0, confounds will be standardized using the sample std instead of the population std.
  cleaned_img = clean_img(fmri_path, detrend = False, standardize = True, confounds = confound_matrix)
C:\Users\kanno\AppData\Local\Temp\ipykernel_5092\1823858860.py:74: FutureWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the default strategy will be replaced by the new strategy, the 'zscore


Processing: sub-04_task-sleep_run3_space-MNI152NLin6Asym_desc-filtered_bold.nii.gz


C:\Users\kanno\AppData\Local\Temp\ipykernel_5092\1823858860.py:74: FutureWarning: boolean values for 'standardize' will be deprecated in nilearn 0.15.0.
Use 'zscore_sample' instead of 'True' or use 'None' instead of 'False'.
  cleaned_img = clean_img(fmri_path, detrend = False, standardize = True, confounds = confound_matrix)
C:\Users\kanno\AppData\Local\Temp\ipykernel_5092\1823858860.py:74: DeprecationWarning: From release 0.14.0, confounds will be standardized using the sample std instead of the population std.
  cleaned_img = clean_img(fmri_path, detrend = False, standardize = True, confounds = confound_matrix)
C:\Users\kanno\AppData\Local\Temp\ipykernel_5092\1823858860.py:74: FutureWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the default strategy will be replaced by the new strategy, the 'zscore

... (252 outputs left out)


Processing: sub-33_task-sleep_run6_space-MNI152NLin6Asym_desc-filtered_bold.nii.gz


C:\Users\kanno\AppData\Local\Temp\ipykernel_5092\1823858860.py:74: FutureWarning: boolean values for 'standardize' will be deprecated in nilearn 0.15.0.
Use 'zscore_sample' instead of 'True' or use 'None' instead of 'False'.
  cleaned_img = clean_img(fmri_path, detrend = False, standardize = True, confounds = confound_matrix)
C:\Users\kanno\AppData\Local\Temp\ipykernel_5092\1823858860.py:74: DeprecationWarning: From release 0.14.0, confounds will be standardized using the sample std instead of the population std.
  cleaned_img = clean_img(fmri_path, detrend = False, standardize = True, confounds = confound_matrix)
C:\Users\kanno\AppData\Local\Temp\ipykernel_5092\1823858860.py:74: FutureWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the default strategy will be replaced by the new strategy, the 'zscore


Processing: sub-33_task-sleep_run7_space-MNI152NLin6Asym_desc-filtered_bold.nii.gz


C:\Users\kanno\AppData\Local\Temp\ipykernel_5092\1823858860.py:74: FutureWarning: boolean values for 'standardize' will be deprecated in nilearn 0.15.0.
Use 'zscore_sample' instead of 'True' or use 'None' instead of 'False'.
  cleaned_img = clean_img(fmri_path, detrend = False, standardize = True, confounds = confound_matrix)
C:\Users\kanno\AppData\Local\Temp\ipykernel_5092\1823858860.py:74: DeprecationWarning: From release 0.14.0, confounds will be standardized using the sample std instead of the population std.
  cleaned_img = clean_img(fmri_path, detrend = False, standardize = True, confounds = confound_matrix)
C:\Users\kanno\AppData\Local\Temp\ipykernel_5092\1823858860.py:74: FutureWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the default strategy will be replaced by the new strategy, the 'zscore

In [10]:
wake_fc = np.load("../4_results/1_arousal/1_EEG_Sleep/EEG_rest_events.npy")
sleep_fc = np.load("../4_results/1_arousal/1_EEG_Sleep/EEG_sleep_events.npy")

fmri_files = sorted([f for f in os.listdir('../1_data/1_EEG_Sleep_fMRI/') if f.endswith(".nii.gz")])
subjs = []
for file in fmri_files:
    subj = file.split("_")[0]
    subjs.append(subj)


subjs = list(set(subjs))
subjs = sorted(subjs)
if "sub-03" in subjs:
    subjs.remove("sub-03")
if "sub-09" in subjs:
    subjs.remove("sub-09")

fmri_dir = "../1_data/1_EEG_Sleep_fMRI/"
fmri_files = sorted([f for f in os.listdir(fmri_dir) if f.endswith(".nii.gz")])

runs_per_subject = {}

for file in fmri_files:
    subj = file.split("_")[0]
    run = ''.join([char for char in file.split("_")[2] if char.isdigit()])
    if subj not in runs_per_subject:
        runs_per_subject[subj] = []
    runs_per_subject[subj].append(run)

counter = 0
sub_counter = -1 
for subject in subjs:
    sub_counter += 1
    if subject in runs_per_subject:
        n_runs = len(runs_per_subject[subject])
        print(f"{subject} has {n_runs} runs.")
        this_wake = wake_fc[sub_counter,:,:]
        this_sleep = sleep_fc[counter:counter + n_runs,:,:]
        sleep_reshaped = this_sleep.reshape(30*n_runs, 35778)
        combined_fc = np.concatenate((this_wake, sleep_reshaped), axis=0)
        counter += n_runs


        print(f"Saving combined FC for {subject} with shape {combined_fc.shape}")
        np.save(f"../4_results/1_arousal/1_EEG_Sleep/combined_sleepwake/{subject}_combined_fc.npy", combined_fc)

    else: 
        print(f"No sleep runs found for {subject}.")
        this_wake = wake_fc[sub_counter,:,:]
        np.save(f"../4_results/1_arousal/1_EEG_Sleep/combined_sleepwake/{subject}_combined_fc.npy", this_wake)

wake_files = sorted([f for f in os.listdir('../1_data/1_EEG_Sleep_fMRI/wakestage') if f.endswith(".tsv")])
wakemap = {"W":4,"W (uncertain)":4, "1":3,"1 (uncertain)":3, "2":2,"2 (uncertain)":2, "3":1, "3 (uncertain)":1}

for i, file in enumerate(wake_files):
    sub = file.split("-")[1]
    waketsv = pd.read_csv(os.path.join('../1_data/1_EEG_Sleep_fMRI/wakestage', file), sep="\t")['30-sec_epoch_sleep_stage']
    this_wake = waketsv.map(wakemap).values
    np.save(f"../4_results/1_arousal/1_EEG_Sleep/combined_sleepwake/sub-{sub}_wake.npy", this_wake)

wake_files = sorted([f for f in os.listdir('../1_data/1_EEG_Sleep_fMRI/wakestage') if f.endswith(".tsv")])
wake_data = np.empty((n_subjects, 40))

wakemap = {"W":4,"W (uncertain)":4, "1":3,"1 (uncertain)":3, "2":2,"2 (uncertain)":2, "3":1, "3 (uncertain)":1}

remove_from_wake_files = ['sub-01-sleep-stage.tsv', 'sub-03-sleep-stage.tsv', 'sub-09-sleep-stage.tsv', 'sub-15-sleep-stage.tsv', 'sub-16-sleep-stage.tsv',
                          'sub-24-sleep-stage.tsv', 'sub-27-sleep-stage.tsv', 'sub-30-sleep-stage.tsv', 'sub-32-sleep-stage.tsv']

wake_files = [f for f in wake_files if f not in remove_from_wake_files]

for i, file in enumerate(wake_files):
    waketsv = pd.read_csv(os.path.join('../1_data/1_EEG_Sleep_fMRI/wakestage', file), sep="\t")['30-sec_epoch_sleep_stage'][0:40]
    wake_data[i,:] = waketsv.map(wakemap).values

np.save("../4_results/1_arousal/1_EEG_Sleep/combined_sleepwake/wakestage.npy", wake_data)

sub-04 has 7 runs.
Saving combined FC for sub-04 with shape (250, 35778)
sub-05 has 6 runs.
Saving combined FC for sub-05 with shape (220, 35778)
sub-06 has 4 runs.
Saving combined FC for sub-06 with shape (160, 35778)
sub-07 has 5 runs.
Saving combined FC for sub-07 with shape (190, 35778)
sub-08 has 7 runs.
Saving combined FC for sub-08 with shape (250, 35778)
sub-10 has 6 runs.
Saving combined FC for sub-10 with shape (220, 35778)
sub-11 has 6 runs.
Saving combined FC for sub-11 with shape (220, 35778)
sub-12 has 3 runs.
Saving combined FC for sub-12 with shape (130, 35778)
sub-13 has 4 runs.
Saving combined FC for sub-13 with shape (160, 35778)
sub-14 has 7 runs.
Saving combined FC for sub-14 with shape (250, 35778)
sub-17 has 7 runs.
Saving combined FC for sub-17 with shape (250, 35778)
sub-18 has 5 runs.
Saving combined FC for sub-18 with shape (190, 35778)
sub-19 has 5 runs.
Saving combined FC for sub-19 with shape (190, 35778)
sub-20 has 5 runs.
Saving combined FC for sub-20 wi

In [28]:
EEGWS_fc = np.load("../4_results/1_arousal/1_EEG_Sleep/EEG_rest_events.npy")
EEGWS_arousal = np.load("../4_results/1_arousal/1_EEG_Sleep/combined_sleepwake/wakestage.npy")
idx = np.where((EEGWS_arousal == EEGWS_arousal[:, [0]]).all(axis=1))[0]

EEGWS_arousal = np.delete(EEGWS_arousal, idx, axis=0)
EEGWS_fc = np.delete(EEGWS_fc, idx, axis=0)

eeg_fc_array_cut = np.load("../4_results/1_arousal/1_EEG_Rest/EEG_Rest_FC_Cut.npy")
eeg_array_cut = np.load("../4_results/1_arousal/1_EEG_Rest/EEG_Rest_Arousal_Cut.npy")
pupil_fc_array_cut = np.load("../4_results/1_arousal/2_Pupil_Rest/Pupil_Rest_FC_Cut.npy")
pupil_array_cut = np.load("../4_results/1_arousal/2_Pupil_Rest/Pupil_Rest_Arousal_Cut.npy")
paranoia_fc_array_cut = np.load("../4_results/1_arousal/3_Pupil_Story/Pupil_Story_FC_Cut.npy")
paranoia_array_cut = np.load("../4_results/1_arousal/3_Pupil_Story/Pupil_Story_Arousal_Cut.npy")
FF_event_array_cut = np.load("../4_results/1_arousal/4_FilmFestival/FilmFestival_Arousal_Cut.npy")
FF_event_fc_array_cut = np.load("../4_results/1_arousal/4_FilmFestival/FilmFestival_FC_Cut.npy")
Sherlock_event_array_cut = np.load("../4_results/1_arousal/5_Sherlock/Sherlock_Arousal_Cut.npy")
Sherlock_event_fc_array_cut = np.load("../4_results/1_arousal/5_Sherlock/Sherlock_FC_Cut.npy")

In [18]:
arousal1_array = EEGWS_arousal
fc1_array = EEGWS_fc
arousal2_array = pupil_array_cut
fc2_array = pupil_fc_array_cut

predicted_output_path = "../4_results/2_classification/16_Wakestage_from_Pupil_Rest/Wakestage_from_Pupil_Rest_predicted.npy"
observed_output_path = "../4_results/2_classification/16_Wakestage_from_Pupil_Rest/Wakestage_from_Pupil_Rest_observed.npy"
features_output_path = "../4_results/2_classification/16_Wakestage_from_Pupil_Rest/Wakestage_from_Pupil_Rest_features.csv"

cross_modal_prediction(arousal1_array, fc1_array, arousal2_array, fc2_array,
                           predicted_output_path, observed_output_path, features_output_path, alpha_threshold = 0.05, 
                           use_mean=True, use_binary_weights=True,
                           normalize_by_total=True, use_absolute=False)


Shape of Arousal 1 data: (19, 40)
Shape of Arousal 1 FC data: (19, 40, 35778)
Shape of Arousal 2 data: (25, 740)
Shape of Arousal 2 FC data: (25, 740, 35778)
Extracting features from Arousal 2 data...
Running one-sample t-test on Arousal 2 features...
[ 1.  1.  1. ... -1.  1. -1.]
4815 significant features selected (p < 0.05)
Number of positive features: 3682
Number of negative features: 1133
Total number of features: 4815
Mapping significant features to ROI pairs...
Applying Arousal 2-derived features to Arousal 1 data...
Saving cross-modal prediction results...
✅ Cross-modal prediction completed! 🚀


In [19]:
arousal1_array = EEGWS_arousal
fc1_array = EEGWS_fc
arousal2_array = paranoia_array_cut
fc2_array = paranoia_fc_array_cut

predicted_output_path = "../4_results/2_classification/17_Wakestage_from_Pupil_Story/Wakestage_from_Pupil_Story_predicted.npy"
observed_output_path = "../4_results/2_classification/17_Wakestage_from_Pupil_Story/Wakestage_from_Pupil_Story_observed.npy"
features_output_path = "../4_results/2_classification/17_Wakestage_from_Pupil_Story/Wakestage_from_Pupil_Story_features.csv"

cross_modal_prediction(arousal1_array, fc1_array, arousal2_array, fc2_array,
                           predicted_output_path, observed_output_path, features_output_path, alpha_threshold = 0.05, 
                           use_mean=True, use_binary_weights=True,
                           normalize_by_total=True, use_absolute=False)

Shape of Arousal 1 data: (19, 40)
Shape of Arousal 1 FC data: (19, 40, 35778)
Shape of Arousal 2 data: (20, 1212)
Shape of Arousal 2 FC data: (20, 1212, 35778)
Extracting features from Arousal 2 data...
Running one-sample t-test on Arousal 2 features...
[-1. -1.  1. ... -1. -1.  1.]
3457 significant features selected (p < 0.05)
Number of positive features: 2240
Number of negative features: 1217
Total number of features: 3457
Mapping significant features to ROI pairs...
Applying Arousal 2-derived features to Arousal 1 data...
Saving cross-modal prediction results...
✅ Cross-modal prediction completed! 🚀


In [20]:
arousal1_array = EEGWS_arousal
fc1_array = EEGWS_fc
arousal2_array = FF_event_array_cut
fc2_array = FF_event_fc_array_cut

predicted_output_path = "../4_results/2_classification/18_Wakestage_from_Movies/Wakestage_from_FF_predicted.npy"
observed_output_path = "../4_results/2_classification/18_Wakestage_from_Movies/Wakestage_from_FF_observed.npy"
features_output_path = "../4_results/2_classification/18_Wakestage_from_Movies/Wakestage_from_FF_features.csv"

cross_modal_prediction(arousal1_array, fc1_array, arousal2_array, fc2_array,
                           predicted_output_path, observed_output_path, features_output_path, alpha_threshold = 0.05, 
                           use_mean=True, use_binary_weights=True,
                           normalize_by_total=True, use_absolute=False)

arousal1_array = EEGWS_arousal
fc1_array = EEGWS_fc
arousal2_array = Sherlock_event_array_cut
fc2_array = Sherlock_event_fc_array_cut

predicted_output_path = "../4_results/2_classification/18_Wakestage_from_Movies/Wakestage_from_Sherlock_predicted.npy"
observed_output_path = "../4_results/2_classification/18_Wakestage_from_Movies/Wakestage_from_Sherlock_observed.npy"
features_output_path = "../4_results/2_classification/18_Wakestage_from_Movies/Wakestage_from_Sherlock_features.csv"

cross_modal_prediction(arousal1_array, fc1_array, arousal2_array, fc2_array,
                           predicted_output_path, observed_output_path, features_output_path, alpha_threshold = 0.05, 
                           use_mean=True, use_binary_weights=True,
                           normalize_by_total=True, use_absolute=False)

Shape of Arousal 1 data: (19, 40)
Shape of Arousal 1 FC data: (19, 40, 35778)
Shape of Arousal 2 data: (15, 68)
Shape of Arousal 2 FC data: (15, 68, 35778)
Extracting features from Arousal 2 data...


C:\Users\kanno\AppData\Local\Temp\ipykernel_5092\3214571245.py:49: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, _ = stats.pearsonr(fc2_clean[:, feature_idx], arousal2_clean)


Running one-sample t-test on Arousal 2 features...
[-1. -1. -1. ... nan -1. -1.]
6531 significant features selected (p < 0.05)
Number of positive features: 3734
Number of negative features: 2239
Total number of features: 6531
Mapping significant features to ROI pairs...
Applying Arousal 2-derived features to Arousal 1 data...
Saving cross-modal prediction results...
✅ Cross-modal prediction completed! 🚀
Shape of Arousal 1 data: (19, 40)
Shape of Arousal 1 FC data: (19, 40, 35778)
Shape of Arousal 2 data: (14, 48)
Shape of Arousal 2 FC data: (14, 48, 35778)
Extracting features from Arousal 2 data...
Running one-sample t-test on Arousal 2 features...


C:\Users\kanno\AppData\Local\Temp\ipykernel_5092\3214571245.py:54: SmallSampleWarning: After omitting NaNs, one or more axis-slices of one or more sample arguments is too small; corresponding elements of returned arrays will be NaN. See documentation for sample size requirements.
  t_values, p_values = stats.ttest_1samp(r_matrix, popmean=0, axis=0, nan_policy='omit')
c:\Users\kanno\anaconda3\Lib\site-packages\scipy\stats\_stats_py.py:1214: RuntimeWarning: divide by zero encountered in divide
  var *= np.divide(n, n-ddof)  # to avoid error on division by zero
c:\Users\kanno\anaconda3\Lib\site-packages\scipy\stats\_stats_py.py:1214: RuntimeWarning: invalid value encountered in scalar multiply
  var *= np.divide(n, n-ddof)  # to avoid error on division by zero


[1. 1. 1. ... 1. 1. 1.]
4412 significant features selected (p < 0.05)
Number of positive features: 3093
Number of negative features: 1111
Total number of features: 4412
Mapping significant features to ROI pairs...
Applying Arousal 2-derived features to Arousal 1 data...
Saving cross-modal prediction results...
✅ Cross-modal prediction completed! 🚀


In [26]:
pupil_to_sw_r, pupil_to_sw_p, pupil_to_sw_predicted, sw_observed, pupil_to_sw_iters, pupil_to_sw_corrs = predict_sleepwake(pupil_array_cut, pupil_fc_array_cut, alpha_threshold = 0.05, 
                           use_mean=True, use_binary_weights=True,
                           normalize_by_total=True, use_absolute=False)

np.save("../4_results/2_classification/16_Wakestage_from_Pupil_Rest/Wakestage_Sleep_from_Pupil_Rest_predicted.npy", pupil_to_sw_predicted)
np.save("../4_results/2_classification/16_Wakestage_from_Pupil_Rest/Wakestage_Sleep_from_Pupil_Rest_observed.npy", sw_observed)
np.save("../4_results/2_classification/16_Wakestage_from_Pupil_Rest/Wakestage_Sleep_from_Pupil_Rest_iters.npy", pupil_to_sw_iters)
np.save("../4_results/2_classification/16_Wakestage_from_Pupil_Rest/Wakestage_Sleep_from_Pupil_Rest_r.npy", pupil_to_sw_r)
np.save("../4_results/2_classification/16_Wakestage_from_Pupil_Rest/Wakestage_Sleep_from_Pupil_Rest_p.npy", pupil_to_sw_p)
np.save("../4_results/2_classification/16_Wakestage_from_Pupil_Rest/Wakestage_Sleep_from_Pupil_Rest_corrs.npy", pupil_to_sw_corrs)

Shape of Arousal 2 data: (25, 740)
Shape of Arousal 2 FC data: (25, 740, 35778)
Extracting features from Arousal 2 data...
Running one-sample t-test on Arousal 2 features...
[ 1.  1.  1. ... -1.  1. -1.]
Mean correlation across all subjects: 0.12420785225833803
P-value from permutation test: 0.0


In [30]:
eeg_to_sw_r, eeg_to_sw_p, eeg_to_sw_predicted, sw_observed, eeg_to_sw_iters, eeg_to_sw_corrs = predict_sleepwake(eeg_array_cut, eeg_fc_array_cut, alpha_threshold = 0.05, 
                           use_mean=True, use_binary_weights=True,
                           normalize_by_total=True, use_absolute=False)

np.save("../4_results/2_classification/19_Wakestage_from_EEG_Rest/Wakestage_Sleep_from_EEG_Rest_predicted.npy", eeg_to_sw_predicted)
np.save("../4_results/2_classification/19_Wakestage_from_EEG_Rest/Wakestage_Sleep_from_EEG_Rest_observed.npy", sw_observed)
np.save("../4_results/2_classification/19_Wakestage_from_EEG_Rest/Wakestage_Sleep_from_EEG_Rest_corrs.npy", eeg_to_sw_corrs)
np.save("../4_results/2_classification/19_Wakestage_from_EEG_Rest/Wakestage_Sleep_from_EEG_Rest_iters.npy", eeg_to_sw_iters)
np.save("../4_results/2_classification/19_Wakestage_from_EEG_Rest/Wakestage_Sleep_from_EEG_Rest_r.npy", eeg_to_sw_r)
np.save("../4_results/2_classification/19_Wakestage_from_EEG_Rest/Wakestage_Sleep_from_EEG_Rest_p.npy", eeg_to_sw_p)

Shape of Arousal 2 data: (24, 544)
Shape of Arousal 2 FC data: (24, 544, 35778)
Extracting features from Arousal 2 data...
Running one-sample t-test on Arousal 2 features...
[ 1. -1. -1. ... -1. -1.  1.]
Mean correlation across all subjects: 0.13576092789425484
P-value from permutation test: 0.0


In [31]:
paranoia_to_sw_r, paranoia_to_sw_p, paranoia_to_sw_predicted, sw_observed, paranoia_to_sw_iters, paranoia_to_sw_corrs = predict_sleepwake(paranoia_array_cut, paranoia_fc_array_cut, alpha_threshold = 0.05, 
                           use_mean=True, use_binary_weights=True,
                           normalize_by_total=True, use_absolute=False)

np.save("../4_results/2_classification/17_Wakestage_from_Pupil_Story/Wakestage_Sleep_from_Pupil_Story_predicted.npy", paranoia_to_sw_predicted)
np.save("../4_results/2_classification/17_Wakestage_from_Pupil_Story/Wakestage_Sleep_from_Pupil_Story_observed.npy", sw_observed)
np.save("../4_results/2_classification/17_Wakestage_from_Pupil_Story/Wakestage_Sleep_from_Pupil_Story_corrs.npy", paranoia_to_sw_corrs)
np.save("../4_results/2_classification/17_Wakestage_from_Pupil_Story/Wakestage_Sleep_from_Pupil_Story_iters.npy", paranoia_to_sw_iters)
np.save("../4_results/2_classification/17_Wakestage_from_Pupil_Story/Wakestage_Sleep_from_Pupil_Story_r.npy", paranoia_to_sw_r)
np.save("../4_results/2_classification/17_Wakestage_from_Pupil_Story/Wakestage_Sleep_from_Pupil_Story_p.npy", paranoia_to_sw_p)

Shape of Arousal 2 data: (20, 1212)
Shape of Arousal 2 FC data: (20, 1212, 35778)
Extracting features from Arousal 2 data...
Running one-sample t-test on Arousal 2 features...
[-1. -1.  1. ... -1. -1.  1.]
Mean correlation across all subjects: 0.1143343873118157
P-value from permutation test: 0.0


In [41]:
df_sig, SR_features, SR_weights = select_and_save_significant_features_v2(
    Sherlock_event_array_cut, Sherlock_event_fc_array_cut,
    FF_event_array_cut, FF_event_fc_array_cut,
    alpha_threshold=0.05,
    use_mean=True
)
    
SR_binarized = np.sign(SR_weights[SR_features])

SR_to_sw_r, SR_to_sw_p, SR_to_sw_predicted, sw_observed, SR_to_sw_iters, SR_to_sw_corrs = SR_to_sleepwake(significant_features = SR_features, weight_mask = SR_binarized)

np.save("../4_results/2_classification/18_Wakestage_from_Movies/Wakestage_Sleep_from_Movies_predicted.npy", SR_to_sw_predicted)
np.save("../4_results/2_classification/18_Wakestage_from_Movies/Wakestage_Sleep_from_Movies_observed.npy", sw_observed)
np.save("../4_results/2_classification/18_Wakestage_from_Movies/Wakestage_Sleep_from_Movies_iters.npy", SR_to_sw_iters)
np.save("../4_results/2_classification/18_Wakestage_from_Movies/Wakestage_Sleep_from_Movies_r.npy", SR_to_sw_r)
np.save("../4_results/2_classification/18_Wakestage_from_Movies/Wakestage_Sleep_from_Movies_corrs.npy", SR_to_sw_corrs)
np.save("../4_results/2_classification/18_Wakestage_from_Movies/Wakestage_Sleep_from_Movies_p.npy", SR_to_sw_p)

Computing per-subject correlations...


C:\Users\kanno\AppData\Local\Temp\ipykernel_5092\1266295292.py:320: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, _ = stats.pearsonr(fc_clean[:, feat_idx], arousal_clean)


Combining datasets...
Running one-sample t-test across all subjects...
6673 features passed p < 0.05
Mapping significant features to ROI pairs...
Mean correlation across all subjects: 0.12999155224728035
P-value from permutation test: 0.0
